In [ ]:
# survey_splitter_app.py
# Desktop Tkinter app:
# - Load raw daily file
# - Preview raw (left) and cleaned (right)
# - Split/export into one Excel workbook per Survey Name into a chosen folder
#
# Handles:
# - .xlsx true Excel
# - .xls true Excel OR TSV/CSV text exports mislabeled as .xls
# - broken quotes in text exports (tolerant parsing)

import io
import os
import re
import csv
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

import pandas as pd


# Clean output template (matches your sample output headers, including leading spaces)
CLEAN_COLS = [
    "Activity Date",
    "Retailer Code",
    "Retailer Name",
    "Retailer Type",
    "Retailer State",
    " Retailer District",
    " Retailer City",
    "Question",
    " Label",
    " Answer",
]


def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())


def make_safe_filename(name: str, max_len: int = 120) -> str:
    # Remove characters invalid for filenames across OSes
    safe = re.sub(r'[<>:"/\\\\|?*]', "-", str(name)).strip()
    safe = re.sub(r"\s+", " ", safe)
    if not safe:
        safe = "UNKNOWN"
    return safe[:max_len]


def read_raw_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()

    # Try Excel first
    if ext == ".xlsx":
        return pd.read_excel(path, engine="openpyxl")

    if ext == ".xls":
        try:
            return pd.read_excel(path)
        except Exception:
            pass  # fall back to tolerant text parsing

    # Fallback: tolerant parsing for TSV/CSV text exports
    with open(path, "rb") as f:
        raw_bytes = f.read()

    text = raw_bytes.decode("utf-8", errors="ignore")

    # TSV first
    try:
        return pd.read_csv(
            io.StringIO(text),
            sep="\t",
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip",
        )
    except Exception:
        # then CSV
        return pd.read_csv(
            io.StringIO(text),
            sep=",",
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip",
        )


def normalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().strip('"').strip("'") for c in df.columns]
    return df


def build_clean_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Takes raw export and returns a cleaned DataFrame with:
    - exact CLEAN_COLS (matching sample output)
    - date parsed
    - text stripped
    Also returns survey name series to support splitting.
    """
    df = normalize_headers(df_raw)

    # Build a lookup of raw columns by normalized names
    raw_map = {_norm(c): c for c in df.columns}

    # We expect Survey Name in raw (sometimes has leading space)
    survey_col = None
    for k in ["survey name", " survey name"]:
        if k in raw_map:
            survey_col = raw_map[k]
            break

    if survey_col is None:
        # try fuzzy contains
        for c in df.columns:
            if "survey" in _norm(c) and "name" in _norm(c):
                survey_col = c
                break

    if survey_col is None:
        raise ValueError("Survey Name column not found in the raw file.")

    # Map raw columns into the clean template
    # Notice the clean template has leading spaces for some headers; we still match using normalized names.
    out = pd.DataFrame()
    for needed in CLEAN_COLS:
        key = _norm(needed)
        # Special-case: raw may have "Retailer District" or " Retailer District" etc.
        if key in raw_map:
            out[needed] = df[raw_map[key]]
        else:
            # fuzzy search
            found = None
            for c in df.columns:
                if _norm(c) == key:
                    found = c
                    break
            out[needed] = df[found] if found else pd.NA

    # Keep Survey Name separately for splitting/export
    survey_series = df[survey_col].astype(str).str.strip()
    survey_series = survey_series.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA})

    # Clean text columns in output
    for c in out.columns:
        if out[c].dtype == "object":
            s = out[c].astype(str)
            s = s.str.replace(r'^\s*"\s*|\s*"\s*$', "", regex=True).str.strip()
            s = s.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA})
            out[c] = s

    # Parse Activity Date
    out["Activity Date"] = pd.to_datetime(out["Activity Date"], errors="coerce", dayfirst=True)

    # Drop fully empty rows
    out = out.dropna(how="all")

    # Ensure exact column order
    out = out[CLEAN_COLS]

    return out, survey_series.loc[out.index]


def export_by_survey(df_clean: pd.DataFrame, survey_series: pd.Series, out_dir: str) -> int:
    """
    Creates one workbook per survey name:
    <Survey Name>.xlsx
    Each workbook has a single sheet named 'Sheet1' (matching your sample output).
    """
    os.makedirs(out_dir, exist_ok=True)

    # Drop rows without survey name
    keep = survey_series.notna()
    df_clean = df_clean.loc[keep].copy()
    survey_series = survey_series.loc[keep].copy()

    count = 0
    for survey_name, sub_idx in survey_series.groupby(survey_series).groups.items():
        sub = df_clean.loc[sub_idx].copy()

        file_name = make_safe_filename(survey_name) + ".xlsx"
        out_path = os.path.join(out_dir, file_name)

        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            sub.to_excel(writer, index=False, sheet_name="Sheet1")

        count += 1

    return count


class DataPreview(ttk.Frame):
    def __init__(self, parent, title: str):
        super().__init__(parent)
        self.title = ttk.Label(self, text=title)
        self.title.pack(anchor="w", padx=6, pady=(6, 2))

        self.tree = ttk.Treeview(self, show="headings")
        self.vsb = ttk.Scrollbar(self, orient="vertical", command=self.tree.yview)
        self.hsb = ttk.Scrollbar(self, orient="horizontal", command=self.tree.xview)
        self.tree.configure(yscrollcommand=self.vsb.set, xscrollcommand=self.hsb.set)

        self.tree.pack(side="top", fill="both", expand=True, padx=6)
        self.vsb.pack(side="right", fill="y")
        self.hsb.pack(side="bottom", fill="x")

        self.note = ttk.Label(self, text="", foreground="#555")
        self.note.pack(anchor="w", padx=6, pady=(2, 6))

    def show_df(self, df: pd.DataFrame, max_rows: int = 50):
        self.tree.delete(*self.tree.get_children())

        cols = list(df.columns)
        self.tree["columns"] = cols

        for c in cols:
            self.tree.heading(c, text=c)
            self.tree.column(c, width=min(max(100, len(str(c)) * 10), 280), anchor="w")

        preview = df.head(max_rows).fillna("")
        for _, row in preview.iterrows():
            self.tree.insert("", "end", values=[str(v) for v in row.values])

        self.note.config(text=f"Showing {min(len(df), max_rows)} of {len(df)} rows")


class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Survey Split Cleaner (Raw → Clean → Export by Survey Name)")
        self.geometry("1250x700")

        self.raw_path = None
        self.df_raw = None
        self.df_clean = None
        self.survey_series = None
        self.surveys = []

        top = ttk.Frame(self)
        top.pack(fill="x", padx=10, pady=10)

        self.btn_load = ttk.Button(top, text="1) Load Raw File", command=self.load_file)
        self.btn_load.pack(side="left")

        self.btn_clean = ttk.Button(top, text="2) Run Cleaning", command=self.run_cleaning, state="disabled")
        self.btn_clean.pack(side="left", padx=(10, 0))

        self.btn_export = ttk.Button(top, text="3) Export Workbooks (per Survey)", command=self.export_folder, state="disabled")
        self.btn_export.pack(side="left", padx=(10, 0))

        ttk.Label(top, text="Preview Survey:").pack(side="left", padx=(20, 6))

        self.survey_var = tk.StringVar(value="")
        self.survey_combo = ttk.Combobox(top, textvariable=self.survey_var, state="disabled", width=40)
        self.survey_combo.pack(side="left")
        self.survey_combo.bind("<<ComboboxSelected>>", self.on_survey_selected)

        self.status = ttk.Label(top, text="No file loaded.", foreground="#333")
        self.status.pack(side="left", padx=14)

        main = ttk.Frame(self)
        main.pack(fill="both", expand=True, padx=10, pady=(0, 10))

        self.left = DataPreview(main, "Input (Raw)")
        self.right = DataPreview(main, "Output (Cleaned)")

        self.left.pack(side="left", fill="both", expand=True)
        self.right.pack(side="left", fill="both", expand=True, padx=(10, 0))

        try:
            style = ttk.Style()
            style.theme_use("clam")
        except Exception:
            pass

        self.right.show_df(pd.DataFrame(columns=CLEAN_COLS))

    def load_file(self):
        path = filedialog.askopenfilename(
            title="Select Raw File",
            filetypes=[
                ("Excel/Export Files", "*.xls *.xlsx *.csv *.txt"),
                ("All Files", "*.*"),
            ],
        )
        if not path:
            return

        try:
            df = read_raw_file(path)
            self.raw_path = path
            self.df_raw = df
            self.df_clean = None
            self.survey_series = None
            self.surveys = []

            self.left.show_df(df)
            self.right.show_df(pd.DataFrame(columns=CLEAN_COLS))

            self.btn_clean.config(state="normal")
            self.btn_export.config(state="disabled")

            self.survey_combo.config(state="disabled", values=[])
            self.survey_var.set("")

            self.status.config(text=f"Loaded: {os.path.basename(path)}")
        except Exception as e:
            messagebox.showerror("Load Error", f"Failed to load file:\n{e}")

    def run_cleaning(self):
        if self.df_raw is None:
            return
        try:
            df_clean, survey_series = build_clean_df(self.df_raw)
            self.df_clean = df_clean
            self.survey_series = survey_series

            # build survey list
            surveys = survey_series.dropna().astype(str).str.strip().unique().tolist()
            surveys = sorted([s for s in surveys if s])

            self.surveys = surveys
            self.survey_combo.config(state="readonly", values=surveys)

            # default preview = first survey
            if surveys:
                self.survey_var.set(surveys[0])
                self.preview_survey(surveys[0])
            else:
                self.right.show_df(df_clean)

            self.btn_export.config(state="normal")

            self.status.config(text=f"Cleaned OK. Surveys detected: {len(surveys)}. Ready to export.")
        except Exception as e:
            messagebox.showerror("Cleaning Error", f"Cleaning failed:\n{e}")

    def on_survey_selected(self, _event):
        name = self.survey_var.get().strip()
        if name:
            self.preview_survey(name)

    def preview_survey(self, survey_name: str):
        if self.df_clean is None or self.survey_series is None:
            return
        mask = self.survey_series.astype(str).str.strip() == survey_name
        sub = self.df_clean.loc[mask].copy()
        self.right.show_df(sub)

        self.status.config(text=f"Preview: {survey_name} | rows: {len(sub)} | total surveys: {len(self.surveys)}")

    def export_folder(self):
        if self.df_clean is None or self.survey_series is None:
            return

        out_dir = filedialog.askdirectory(title="Select Output Folder (Workbooks will be saved here)")
        if not out_dir:
            return

        try:
            n = export_by_survey(self.df_clean, self.survey_series, out_dir)
            messagebox.showinfo(
                "Export Complete",
                f"Exported {n} workbook(s) into:\n{out_dir}\n\nEach workbook contains a Sheet1 with cleaned data.",
            )
            self.status.config(text=f"Exported {n} workbooks to folder.")
        except Exception as e:
            messagebox.showerror("Export Error", f"Export failed:\n{e}")


if __name__ == "__main__":
    App().mainloop()
